In [ ]:
import struct, time, os, numpy as np
from pynq import Overlay, MMIO, allocate

ol = Overlay('resizer.bit')
mmio = MMIO(0x43C00000, 0x10000)

CFG_ENABLE_WORD = 0
CFG_THRESH_BASE = 1152
CFG_SIGN_BASE = 1408
MVAU_OUT_CH = {1: 64, 2: 128, 3: 128, 4: 256, 5: 256}

def mvau_byte_addr(mvau_id, word_addr):
    return (mvau_id << 13) | (word_addr << 2)

def load_bin_u32(path):
    with open(path, 'rb') as f:
        data = f.read()
    return list(struct.unpack(f'<{len(data)//4}I', data))

# DMA setup
idma = ol.idma0
odma = ol.odma0
ibuf = allocate(shape=(1, 32, 32, 3, 1), dtype=np.uint8, cacheable=True)
obuf = allocate(shape=(1, 1, 1), dtype=np.uint8, cacheable=True)

deploy_dir = '/home/xilinx/jupyter_notebooks/finn-cnv-test/pynq_deployment_zl8sy1tn'
img = np.load(deploy_dir + '/input.npy')
ibuf[:] = img.reshape(1, 32, 32, 3, 1)

def run_single():
    ibuf.flush()
    obuf[:] = 0
    obuf.flush()
    odma.write(0x10, obuf.device_address)
    odma.write(0x1C, 1)
    odma.write(0x00, 1)
    idma.write(0x10, ibuf.device_address)
    idma.write(0x1C, 1)
    idma.write(0x00, 1)
    for _ in range(200000):
        if odma.read(0x00) & 0x2:
            break
    obuf.invalidate()
    return int(np.array(obuf).flatten()[0])

# Load all threshold data
cifar10_thresh = {}
svhn_thresh = {}
svhn_sign = {}
for m in range(1, 6):
    cifar10_thresh[m] = load_bin_u32(f'/home/xilinx/runtime_switch/mvau{m}_thresh_cifar10.bin')
    svhn_thresh[m] = load_bin_u32(f'/home/xilinx/runtime_switch/mvau{m}_thresh_svhn.bin')
    svhn_sign[m] = load_bin_u32(f'/home/xilinx/runtime_switch/mvau{m}_sign_svhn.bin')

print('Data loaded successfully')

# Test 0: ROM defaults, adapter OFF
print('\n--- Test 0: ROM defaults, adapter OFF ---')
for m in range(1, 6):
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 0)
pred = run_single()
print(f'  pred={pred} (ROM=SVHN-fused, adapter OFF)')

# Test 1: SVHN mode
print('\n--- Test 1: SVHN mode ---')
for m in range(1, 6):
    for i, val in enumerate(svhn_thresh[m]):
        mmio.write(mvau_byte_addr(m, CFG_THRESH_BASE + i), val)
    for i, val in enumerate(svhn_sign[m]):
        mmio.write(mvau_byte_addr(m, CFG_SIGN_BASE + i), val)
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 1)
pred = run_single()
print(f'  pred={pred} (SVHN mode)')

# Test 2: CIFAR-10 mode
print('\n--- Test 2: CIFAR-10 mode ---')
for m in range(1, 6):
    for i, val in enumerate(cifar10_thresh[m]):
        mmio.write(mvau_byte_addr(m, CFG_THRESH_BASE + i), val)
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 0)
pred = run_single()
print(f'  pred={pred} (CIFAR-10 mode, expected 3)')

# Test 3: Reload and CIFAR-10
print('\n--- Test 3: Reload overlay + CIFAR-10 ---')
ol = Overlay('resizer.bit')
mmio = MMIO(0x43C00000, 0x10000)
idma = ol.idma0
odma = ol.odma0
for m in range(1, 6):
    for i, val in enumerate(cifar10_thresh[m]):
        mmio.write(mvau_byte_addr(m, CFG_THRESH_BASE + i), val)
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 0)
pred = run_single()
print(f'  pred={pred} (fresh reload + CIFAR-10, expected 3)')

# Test 4: Progressive
print('\n--- Test 4: Progressive switch ---')
for m in range(1, 6):
    for i, val in enumerate(svhn_thresh[m]):
        mmio.write(mvau_byte_addr(m, CFG_THRESH_BASE + i), val)
    for i, val in enumerate(svhn_sign[m]):
        mmio.write(mvau_byte_addr(m, CFG_SIGN_BASE + i), val)
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 1)
pred = run_single()
print(f'  All SVHN: pred={pred}')
for switch_m in range(1, 6):
    for i, val in enumerate(cifar10_thresh[switch_m]):
        mmio.write(mvau_byte_addr(switch_m, CFG_THRESH_BASE + i), val)
    mmio.write(mvau_byte_addr(switch_m, CFG_ENABLE_WORD), 0)
    pred = run_single()
    print(f'  MVAU1-{switch_m}=CIFAR10: pred={pred}')

# Test 5: Consistency
print('\n--- Test 5: Consistency (5 runs) ---')
preds = [run_single() for _ in range(5)]
print(f'  Predictions: {preds}')

# Test 6: Extreme thresholds
print('\n--- Test 6: All thresh=0 ---')
for m in range(1, 6):
    for i in range(MVAU_OUT_CH[m]):
        mmio.write(mvau_byte_addr(m, CFG_THRESH_BASE + i), 0)
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 0)
pred = run_single()
print(f'  pred={pred} (all 1s output)')

print('\n--- Test 7: All thresh=MAX ---')
for m in range(1, 6):
    for i in range(MVAU_OUT_CH[m]):
        mmio.write(mvau_byte_addr(m, CFG_THRESH_BASE + i), 0x7FFFFFFF)
    mmio.write(mvau_byte_addr(m, CFG_ENABLE_WORD), 0)
pred = run_single()
print(f'  pred={pred} (all 0s output)')

ibuf.freebuffer()
obuf.freebuffer()
print('\nDiagnostic complete!')
